In [0]:
%sql
DESCRIBE VOLUME practice.default.data_source;

In [0]:
%sql
-- Create source table with the .csv data
CREATE OR REPLACE TABLE practice.default.sales AS
SELECT *
FROM read_files(
    '/Volumes/practice/default/data_source/Sales_Dataset4.csv',
    format => 'csv',
    header => true
);

In [0]:
%sql
-- View the new table
SELECT * 
FROM practice.default.sales 
ORDER BY OrderId DESC
LIMIT 2;

In [0]:
# Load the sales dataset .csv file into a Spark DataFrame
# df = spark.read.format('csv') \
#     .option('header', 'true') \
#     .load('/Volumes/practice/default/data_source/Sales_Dataset.csv')

In [0]:
# Remove the space from the columns
# df = df.toDF(*[col.replace(' ', '_') for col in df.columns])

In [0]:
# Add a load_date to source data (since source data do not have load_date)
# df = df.withColumn('last_updated', lit('2024-01-01'))

In [0]:
# Create a source table sales_data
# df.write.mode('overwrite').saveAsTable('practice.default.sales_data')

In [0]:
# Load the sales_data table into a Spark DataFrame and display the first 5 rows
# df = spark.table('practice.default.sales_data')
# df.toPandas().head(5)

In [0]:
# Load the data in Bronze table based on latest OrderDate (incremental load)

if spark.catalog.tableExists('practice.sales_bronze.b_sales'):
    last_load_date = spark.sql("""
        SELECT MAX(OrderDate) 
        FROM practice.sales_bronze.b_sales
    """).collect()[0][0]
    print('last_load_date:', last_load_date)
else:
    last_load_date = '1000-01-01'
    print('last_load_date:', last_load_date)

In [0]:
# Loading to Bronze table
bronze_df = spark.sql(f"""
    SELECT 
        *,
        CURRENT_TIMESTAMP() AS loaded_at
    FROM practice.default.sales
    WHERE OrderDate > '{last_load_date}'
""")

bronze_df.write.mode('append').saveAsTable('practice.sales_bronze.b_sales')

In [0]:
# Count total records in bronze table
spark.sql("""
    SELECT COUNT(*) 
    FROM practice.sales_bronze.b_sales
""").show()

In [0]:
%sql
-- View sample records from bronze table
SELECT * 
FROM practice.sales_bronze.b_sales 
ORDER BY OrderId DESC
LIMIT 3;